# Notebook 06 - Churn Prediction (Classification)

**Input:** `data/processed/customer_segments.parquet` (5,265 customers x 45 columns)<br>
**Output:**
- `data/processed/churn_predictions.parquet` - each customer with predicted churn probability
- `models/churn_xgboost.joblib` - trained model + metadata
- `reports/figures/churn_*.png` - pitch-ready visualizations

## How this notebook differs from Notebook 05
Notebook 05 predicted **how much** revenue (continous -> regression). Notebook 06 predicts **whether** a customer will buy at all (binary -> classification).<br>
Same modeling framework, different problem shape and different evaluation metrics.

**Why we want both models:**
- CLV regression tells you a customer is worth £450, but doesn't directly answer "will they buy?"
- Churn classification tells you they have 78% probability of buying, but not how much
- Combined: you get the 2x2 matrix that drives the recommendation engine

## Target setup:
- `target_purchased_90d = 1` -> customer bought in next 90 days (active)
- `target_purchased_90d = 0` -> customer did NOT buy (churned)

So we're literally predicting `P(active)`. Churn probability = `1 - P(active)`.

**Class balance:** 43.5% active / 56.5% churned. Mildly imbalanced - we'll use `scale_pos_weight` defensively but it isn't severe enough to require SMOTE or undersampling.

## Notebook structure:
1. Load and prepare features + target
2. Train/test split (stratified on the target)
3. Baseline classifier (majority class)
4. XGBoost classifier
5. Evaluation - ROC, AUC, PR AUC, confusion matrix
6. Probability calibration check
7. Threshold selection - business-driven, not arbitrary
8. SHAP explanations
9. Predictions on full dataset, build the 2x2 strategy matrix with CLV
10. Save model + predictions

## Setup

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, f1_score
)
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports/figures')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE=42